## 第4章 项目结构和代码导入

- **完整项目结构示例：**

```text
omission-git/
│
├── LICENSE.md
├── README.md
├── .gitignore
├── pylintrc
├── omission.py                    # 顶层入口脚本(在顶层包外)
│
└── omission/                      # 顶层包
    ├── __init__.py
    ├── __main__.py                # 包入口点
    ├── app.py
    ├── common/
    │   ├── __init__.py
    │   └── ...
    │
    ├── data/
    │   ├── __init__.py
    │   └── ...
    │
    ├── game/
    │   ├── __init__.py
    │   └── ...
    │
    └── tests/
        ├── __init__.py
        └── ...
```

- **模块和包：**
    - 模块：一个Python(.py)文件。
    - 包：包含一个或多个模块的目录，每个目录都有一个`__init__.py`文件。
        - `__init__.py`文件：包的初始化文件，可以为空，也可以是在包第一次被导入时执行的代码。
    - 命名规则：模块名全小写，下划线可加；包名全小写，下划线尽量不用。

- **导入模块：**
    - `import 模块名`：导入模块，将模块中的代码加载到当前作用域中。
    - `import 模块名 as 别名`：导入模块并换个别名，方便使用(防止命名冲突或方便代码阅读)。
    - `from 模块名 import 函数名, 类名，...`：导入模块中的指定函数、类、变量等。
    - `from 模块名 import 函数名(类名等) as 别名`：导入模块中的函数(类等)并换个别名，方便使用。
    - `from 模块名 import *`：导入模块中所有不以下划线开头的名字。**注意：不建议使用，因为会导入模块中的所有内容，导致命名冲突。**

- **绝对导入和相对导入：**
    - 绝对导入：从根目录开始，逐级向下写的完整路径，推荐使用。
    - 相对导入：从当前模块开始的相对路径，用`.`表示当前目录，`..`表示上一级目录。用`...`表示上两级目录，用`....`表示上三级目录，以此类推。
    - 两种导入方式对比：

| 对比项       | 绝对导入                          | 相对导入                    |
| ------------ | --------------------------------- | --------------------------- |
| 语法示例     | `from omission.common import ...` | `from ...common import ...` |
| 可读性       | ✅ 高，路径完整清晰                | ⚠️ 中，需数 `.` 的个数       |
| 包名变更影响 | 需同步修改路径                    | ✅ 不受影响                  |
| 适用场景     | 所有场景（推荐默认使用）⭐         | 包内部模块间引用            |
| 学习难度     | 低                                | 中（需理解 `.` `..` 含义）  |


- **入口点：**
    - 模块入口：`__name__`属性，当模块导入时，该属性为模块的完全限定名；当模块直接运行时(`python 模块名.py`)，该属性为`__main__`。
        - 因此，通常会在模块末尾添加以下代码：`if __name__ == '__main__':`，用于判断是否为模块直接运行。
    - 包入口：`__main__.py`文件，当包导入时，该文件不会被运行；当包直接运行时(`python -m 包名`)，该文件会被运行。常见代码如下：

        ```python
        def main():
            # 包入口代码
            pass

        if __name__ == '__main__':
            main()
        ```

    - 程序入口：在顶层包的外部创建`main.py`文件，用于程序入口。常见代码如下：

        ```python
        from 包名.__main__ import main
        main()
        ```

- **控制包导入：**
    - 简化导入：在顶层包的`__init__.py`文件中，提前导入相关模块，可以让用户无需记住内部路径，直接从顶层包导入。如：`from .模块名 import 函数名`。
    - 控制import *：在 `__init__.py` 中定义 `__all__` 列表，当遇到 `from 模块名 import *` 时，会把`模块名.__all__`里的每一项替换`*`的位置
        - 例如：`__all__ = ['函数名', '类名', '常量名']`。
        - 注意：`__all__` 里的每一项，必须能合理替换 `from 模块名 import <item>` 中的 `<item>`。

- **模块搜索路径：**
    - 查看搜索路径：`sys.path`。
    - 搜索顺序：先搜索当前目录，再搜索标准库目录，最后搜索第三方库目录。Python按顺序检查每个路径，找到匹配模块就停止。
    - 添加搜索路径：推荐在虚拟环境的`lib/python3.x/site-packages`目录里放一个`.pth`结尾的文件，每行一个路径，绝对路径直接追加，相对路径相对于`.pth`文件位置解析。

- **导入模块机制**：为导入一个模块，Python需要使用两个特殊对象：一个查找器和一个加载器。
    - 查找器：Python导入时先根据元路径查找器(保存在`sys.meta_path`列表中)，查找指定模块。
        - 内置导入器：查找并加载内置模块。
        - 冻结导入器：查找并加载冻结模块(转字节码的模块)。
        - 路径查找器：在文件系统中查找模块。依次尝试每个路径条目查找器(存储在`sys.path_hooks`)，每个路径条目查找器搜索导入路径中列出的每个位置(存储在`sys.path`中)。
    - 加载器：找到模块后，Python会调用加载器加载模块。
        - 通常运行`.py`文件后会生成`.pyc`缓存字节码文件。
        - 加载器检查缓存是否过期：策略1用文件时间戳，策略2用源代码文件哈希值。
        - 加载前会把模块对象加入`sys.modules`(防止导入循环)，再绑定名字到导入者空间。
    - 注意：
        - 如果需要手动导入模块，使用`importlib.import_module()`函数。
        - 无论在项目中导入模块多少次，都只会进行一次查找和加载。

- **核心知识脉络**

```text
第4章 项目结构与导入
│
├── 一、模块与包
│   ├── 模块 = 任何 .py 文件
│   ├── 包 = 含 __init__.py 的目录
│   ├── 命名空间包 = 无 __init__.py 的目录（⚠️ 不能替代传统包）
│   └── 模块本质是对象，包也是模块（多 __path__ 属性）
│
├── 二、PEP 8 命名规范
│   ├── 模块名：全小写，下划线可加
│   └── 包名：全小写，下划线尽量不用
│
├── 三、导入机制
│   ├── import 实际上是运行模块
│   ├── 命名空间：明确函数来源，避免冲突
│   ├── 影子问题：同名函数被覆盖
│   └── 解决方案：as 关键字起别名
│
├── 四、通配符导入 from module import *
│   ├── 导入所有非下划线开头名字
│   ├── 问题：不知道导入了什么 + 命名空间污染
│   └── 黄金法则：❌ 永远不要在生产代码中使用
│
├── 五、绝对导入 vs 相对导入 ⭐
│   ├── 绝对导入：从顶层包写全路径（推荐）✅
│   ├── 相对导入：.（当前包）/ ..（上一级包）/ ...（上两级包）
│   ├── 限制：相对导入只能在包之间跳转
│   └── 同包内导入：from .module import name
│
├── 六、__name__ 属性与入口点 ⭐⭐
│   ├── __name__ = 模块的完全限定名
│   ├── 直接运行时 __name__ = "__main__"
│   ├── if __name__ == "__main__": 标准模式
│   ├── __main__.py：包入口点（python3 -m pkg）
│   └── 入口脚本：omission.py → from omission.__main__ import main
│
├── 七、控制包导入
│   ├── __init__.py 简化导入（选择性重导出）
│   └── __all__ 控制 from pkg import *
│
├── 八、模块搜索路径
│   ├── sys.path 顺序：当前目录 → 标准库 → site-packages
│   ├── 添加路径：.pth 文件（推荐）✅
│   ├── 不要直接修改 sys.path ⚠️
│   └── 底层：sys.meta_path（三个元路径查找器）→ 加载器 → .pyc 缓存
│
└── 九、循环导入
    ├── 定义：A 导入 B，B 导入 A
    ├── 原因：import 时执行整个模块代码
    └── 避免方法：重构消除 / 局部导入 / 合并模块
```

- **警告与提示表**

| 类型     | 内容                                                         |
| ------- | ------------------------------------------------------------ |
| ⚠️ 警告 | 没有 `__init__.py` 的目录是**命名空间包**，不是传统包，两者不可互换！ |
| ⚠️ 警告 | `from module import *` 是**非常糟糕的做法**，极易引发命名冲突 |
| ⚠️ 警告 | 用 `as` 解决命名冲突，如 `from smart_door import open as door_open` |
| ⚠️ 警告 | 相对导入符号：`.` = 当前包，`..` = 父包，不能跨非包目录跳转  |
| ⚠️ 警告 | 同包内导入，如果用 `python3 -m pkg` 运行，绝对导入必须从顶层包开始 |
| ⚠️ 警告 | 不要直接修改 `sys.path` 或 `PYTHONPATH`，用 `.pth` 文件更安全 |
| ⚠️ 警告 | 永远不要从包内其他模块导入你的主模块（会引发循环导入或意外执行） |
| ⚠️ 警告 | 循环导入无法完成初始化——重构消除是根本解法，局部导入是常用 workaround |
| 💡 技巧 | 优先用**绝对导入**，可读性高；相对导入只在包内部简洁使用     |
| 💡 技巧 | 用 `__init__.py` 的 `from .xxx import yyy` 简化用户导入体验  |
| 💡 技巧 | 用 `__all__` 控制 `from pkg import *` 的行为（虽然不推荐用 `*`） |
| 💡 技巧 | 包入口用 `__main__.py` + `if __name__ == "__main__": main()` |
| 💡 技巧 | `.pth` 文件是添加模块搜索路径的正确方式，每行一个绝对/相对路径 |